# Batching Strategies

Online learning algorithms need to handle batched data efficiently. In braintrace, there are two main batching strategies:

- **Map-based batching** (recommended): Wrap single-sample model logic with `brainstate.nn.Map`, then compile from one complete batched time step.
- **Single-sample mode**: Process one sample at a time, without any batching.

The choice of strategy affects how model states are initialized and how the online learning algorithm is called.

This tutorial walks through each strategy with concrete examples and shows how to build a full training loop using Map-based batching.

## Map-Based Batching (Recommended)

The recommended approach is to keep the model's update logic single-sample
and let `brainstate.nn.Map` manage independent state copies across the batch.
`braintrace.compile(..., batch_size=B, vmap=True)` does this in one call:

1. It wraps the model with `brainstate.nn.Map(model, init_map_size=B)`.
2. It initializes the mapped states through `mapped_model.init_all_states()`.
3. It compiles the ETP graph from one batched time step with shape
   `(batch_size, n_in)`.
4. It returns the concrete online-learning algorithm, ready for batched calls.

The returned learner exposes `report` and the rest of the algorithm API
directly.

In [ ]:
import jax
import jax.numpy as jnp
import brainstate
import braintools
import braintrace

In [ ]:
class SimpleGRU(brainstate.nn.Module):
    def __init__(self, n_in, n_rec, n_out):
        super().__init__()
        self.rnn = braintrace.nn.GRUCell(n_in, n_rec)
        self.out = braintrace.nn.Linear(n_rec, n_out)

    def update(self, x):
        return self.out(self.rnn(x))

In [ ]:
model = SimpleGRU(10, 64, 5)
batch_size = 16

# braintrace.compile with vmap=True:
#   - initialises per-sample hidden states (batch_size independent copies)
#   - wraps the model in brainstate.nn.Map and initializes mapped states
#   - compiles the ETP graph from one batched time step
#   - returns the concrete algorithm for parallel mapped execution
mapped_algo = braintrace.compile(
    model, braintrace.D_RTRL, jnp.zeros((batch_size, 10)),
    batch_size=batch_size, vmap=True,
)

# Run on batched input — the returned learner handles the batch axis transparently
x_batch = jnp.ones((batch_size, 10))
out = mapped_algo(x_batch)
print("Output shape:", out.shape)  # (16, 5)

**How it works:**

- `braintrace.compile(..., batch_size=B, vmap=True)` creates
  `brainstate.nn.Map(model, init_map_size=B)` and calls
  `mapped_model.init_all_states()`.
- The algorithm compiles against the batched example input and keeps the ETP
  primitives visible to the compiler.
- Each learner call maps the wrapped model over axis 0 while sharing parameter
  states and maintaining independent recurrent states.

## Single-Sample Mode

For debugging or situations where batch processing is unnecessary, you can compile and run the algorithm on individual samples directly. No `vmap` or state replication is needed.

In [ ]:
model2 = SimpleGRU(10, 64, 5)

# Single-sample mode: omit batch_size so states are created unbatched.
algo2 = braintrace.compile(model2, braintrace.D_RTRL, jnp.zeros(10))

# Process one sample at a time
x_single = jnp.ones(10)
out = algo2(x_single)
print("Single sample output shape:", out.shape)  # (5,)

This mode is straightforward: initialize the model, compile the graph, and call the algorithm. It is useful for step-by-step debugging or when processing a single stream of data.

## Multi-Step Data

braintrace provides `SingleStepData` and `MultiStepData` wrappers to control how the algorithm processes input along the time dimension.

- **`SingleStepData`**: Wraps data for a single time step. The algorithm processes it as one forward pass.
- **`MultiStepData`**: Wraps a sequence of time steps. The algorithm internally scans over all steps in the sequence.

This is useful when you want to pass an entire sequence to the algorithm and have it handle the temporal loop internally, rather than manually iterating over time steps.

In [ ]:
# Single-step: process one time step at a time
x_single = braintrace.SingleStepData(jnp.ones(10))

# Multi-step: process a sequence
sequence = jnp.ones((20, 10))  # 20 time steps, 10 features
x_multi = braintrace.MultiStepData(sequence)

When a `MultiStepData` object is passed to the algorithm, it will iterate over the first axis (time steps) internally. When a `SingleStepData` object (or a plain array) is passed, the algorithm processes it as a single forward step.

## Full Training Loop with Map Batching

Below is a complete example that combines Map-based batching with a temporal training loop. The pattern is:

1. **Map and initialize** independent model states across the batch.
2. **Compile** the algorithm from one batched time step.
3. **Scan** over time steps, accumulating gradients at each step.
4. **Update** parameters with the accumulated gradients.

In [ ]:
@brainstate.transform.jit
def train_step(inputs, targets):
    """inputs: (n_steps, batch_size, n_in), targets: (batch_size,)"""
    # braintrace.compile with vmap=True replaces manual Map initialization and compilation.
    # Pass inputs[0] with shape (batch_size, n_in); the compiler traces the mapped model
    # against the complete batched time step, so do not pass inputs[0, 0].
    mapped_algo = braintrace.compile(
        model, braintrace.D_RTRL, inputs[0],
        batch_size=inputs.shape[1], vmap=True,
    )

    def step_loss(inp):
        out = mapped_algo(inp)
        return jnp.mean((out - targets) ** 2)

    # etrace_grad drives the whole sequence and accumulates the per-step online
    # gradients. Map keeps the batch axis inside the compiled graph, while the
    # learner exposes the same driver methods as the unbatched path.
    return mapped_algo.etrace_grad(inputs, step_fn=step_loss, reduction='sum')

In [ ]:
# Example usage
model = SimpleGRU(10, 64, 5)
inputs = jnp.ones((20, 16, 10))  # 20 steps, batch 16, 10 features
targets = jnp.zeros((16, 5))
grads = train_step(inputs, targets)
print("Gradient keys:", list(grads.keys()))

**What happens in `train_step`:**

1. `braintrace.compile(model, braintrace.D_RTRL, inputs[0], batch_size=B, vmap=True)` wraps the model with `brainstate.nn.Map`, initializes independent per-sample states, compiles from the batched time step, and returns the algorithm directly.
2. `vmapped_algo.etrace_grad` iterates over time, calls `step_fn`, and accumulates online gradients; `reduction='sum'` accumulates without dividing. The call is identical for mapped and directly batched learners.
3. The returned `grads` dictionary keeps the original model parameter paths and can be passed to an optimizer such as `braintools.optim.Adam`.

## Summary

- `braintrace.compile(..., batch_size=B, vmap=True)` is the recommended setup
  for batched online learning.
- Internally it uses `brainstate.nn.Map(model, init_map_size=B)` followed by
  `mapped_model.init_all_states()`.
- The workflow is: compile with `vmap=True`, scan over time, accumulate
  gradients, and update parameters.
- For one stream, compile without `vmap=True`; states remain unbatched.